In [1]:
import math
import numpy as np
import pandas as pd
import QuantLib as ql
from scipy.optimize import brentq

# =============================================================================
# 0.  QuantLib setup  (must match Q5 / Q6 exactly)
# =============================================================================
trade_date = ql.Date(5, 11, 2025)
calendar   = ql.TARGET()
spot_date  = calendar.advance(trade_date, 2, ql.Days)
dc_curve   = ql.Actual365Fixed()
dc_deposit = ql.Actual360()

print(f"Trade date : {trade_date}   Spot date : {spot_date}")

# =============================================================================
# 1.  Load bootstrapped discount curve
# =============================================================================
ts = pd.read_excel("Interp_term_structure.xlsx")
assert "Days" in ts.columns and "DF" in ts.columns

_days_arr = ts["Days"].to_numpy(dtype=float)
_df_arr   = ts["DF"].to_numpy(dtype=float)

def days_from_spot(d: ql.Date) -> int:
    return d.serialNumber() - spot_date.serialNumber()

def get_df(d: ql.Date) -> float:
    """Risk-free discount factor P(spot, d)."""
    n = days_from_spot(d)
    if n <= 0:
        return 1.0
    return float(np.interp(n, _days_arr, _df_arr))

def get_ttm(d: ql.Date) -> float:
    """Act/365 year fraction from spot to d."""
    return dc_curve.yearFraction(spot_date, d)

# =============================================================================
# 2.  Bond constants
# =============================================================================
N          = 1_000.0
ALPHA      = 0.25
CAP_RATE   = 0.0545
FLOOR_RATE = 0.0000
PART       = 1.60
L_STAR     = CAP_RATE / PART

L6 = 0.02029
R6 = min(CAP_RATE, max(FLOOR_RATE, PART * L6))
C6 = N * R6 * ALPHA

# Accrued interest: period 6 start 12 Sep 2025, trade 5 Nov 2025 (30/360)
_days_acc = (2025 - 2025) * 360 + (11 - 9) * 30 + (5 - 12)   # 53 days
AI        = N * R6 * (_days_acc / 360)                         # 4.779

# =============================================================================
# 3.  Payment schedule (periods 6-40)
# =============================================================================
def qldate(d, m, y): return ql.Date(d, m, y)

SCHEDULE = {
     6: qldate(12, 12, 2025),   7: qldate(12,  3, 2026),
     8: qldate(12,  6, 2026),   9: qldate(14,  9, 2026),
    10: qldate(14, 12, 2026),  11: qldate(12,  3, 2027),
    12: qldate(14,  6, 2027),  13: qldate(13,  9, 2027),
    14: qldate(13, 12, 2027),  15: qldate(13,  3, 2028),
    16: qldate(12,  6, 2028),  17: qldate(12,  9, 2028),
    18: qldate(12, 12, 2028),  19: qldate(12,  3, 2029),
    20: qldate(12,  6, 2029),  21: qldate(12,  9, 2029),
    22: qldate(12, 12, 2029),  23: qldate(12,  3, 2030),
    24: qldate(12,  6, 2030),  25: qldate(12,  9, 2030),
    26: qldate(12, 12, 2030),  27: qldate(12,  3, 2031),
    28: qldate(12,  6, 2031),  29: qldate(12,  9, 2031),
    30: qldate(12, 12, 2031),  31: qldate(12,  3, 2032),
    32: qldate(14,  6, 2032),  33: qldate(13,  9, 2032),
    34: qldate(13, 12, 2032),  35: qldate(14,  3, 2033),
    36: qldate(13,  6, 2033),  37: qldate(12,  9, 2033),
    38: qldate(12, 12, 2033),  39: qldate(13,  3, 2034),
    40: qldate(12,  6, 2034),
}

PERIOD_STARTS = {
     6: qldate(12,  9, 2025),   7: qldate(12, 12, 2025),
     8: qldate(12,  3, 2026),   9: qldate(12,  6, 2026),
    10: qldate(14,  9, 2026),  11: qldate(14, 12, 2026),
    12: qldate(12,  3, 2027),  13: qldate(14,  6, 2027),
    14: qldate(13,  9, 2027),  15: qldate(13, 12, 2027),
    16: qldate(13,  3, 2028),  17: qldate(12,  6, 2028),
    18: qldate(12,  9, 2028),  19: qldate(12, 12, 2028),
    20: qldate(12,  3, 2029),  21: qldate(12,  6, 2029),
    22: qldate(12,  9, 2029),  23: qldate(12, 12, 2029),
    24: qldate(12,  3, 2030),  25: qldate(12,  6, 2030),
    26: qldate(12,  9, 2030),  27: qldate(12, 12, 2030),
    28: qldate(12,  3, 2031),  29: qldate(12,  6, 2031),
    30: qldate(12,  9, 2031),  31: qldate(12, 12, 2031),
    32: qldate(12,  3, 2032),  33: qldate(14,  6, 2032),
    34: qldate(13,  9, 2032),  35: qldate(13, 12, 2032),
    36: qldate(14,  3, 2033),  37: qldate(13,  6, 2033),
    38: qldate(12,  9, 2033),  39: qldate(12, 12, 2033),
    40: qldate(13,  3, 2034),
}

MATURITY = SCHEDULE[40]

# =============================================================================
# 4.  Forward EURIBOR and structured coupon helpers
# =============================================================================
def forward_euribor(start: ql.Date, end: ql.Date) -> float:
    alpha = dc_deposit.yearFraction(start, end)
    return (1.0 / alpha) * (get_df(start) / get_df(end) - 1.0)

def structured_coupon_rate(fwd: float) -> float:
    return min(CAP_RATE, max(FLOOR_RATE, PART * fwd))

# =============================================================================
# 5.  Build exposure schedule  (coupons 6-40 + principal)
# =============================================================================
exp_rows = []

for period, pay_date in SCHEDULE.items():
    tau_i = get_ttm(pay_date)
    df_i  = get_df(pay_date)

    if period == 6:
        fwd        = L6
        coup_rate  = R6
        coupon_eur = C6
    else:
        fwd        = forward_euribor(PERIOD_STARTS[period], pay_date)
        coup_rate  = structured_coupon_rate(fwd)
        coupon_eur = N * coup_rate * ALPHA

    exp_rows.append({
        "Period"      : period,
        "Payment Date": pay_date.ISO(),
        "tau_i"       : tau_i,
        "Fwd EURIBOR" : fwd,
        "Coupon Rate" : coup_rate,
        "Exposure"    : coupon_eur,
        "Z_rf"        : df_i,
    })

tau_40 = get_ttm(MATURITY)
df_40  = get_df(MATURITY)
exp_rows.append({
    "Period"      : "Principal",
    "Payment Date": MATURITY.ISO(),
    "tau_i"       : tau_40,
    "Fwd EURIBOR" : np.nan,
    "Coupon Rate" : np.nan,
    "Exposure"    : N,
    "Z_rf"        : df_40,
})

exp_df = pd.DataFrame(exp_rows)

# Risk-free gross NPV  (from Q5 option-adjusted pricing)
# -------------------------------------------------------
# Replace the line below with the actual Q5 option-adjusted
# value once computed; here we use the forward-rate NPV as
# the base for the credit adjustment discussion.
rf_npv_gross = (exp_df["Exposure"] * exp_df["Z_rf"]).sum()
rf_npv_clean = rf_npv_gross - AI

print(f"\nRisk-free NPV  gross : EUR {rf_npv_gross:.4f}  "
      f"({rf_npv_gross / 10:.4f}%)")
print(f"Risk-free NPV  clean : EUR {rf_npv_clean:.4f}  "
      f"({rf_npv_clean / 10:.4f}%)")
print(f"Accrued interest     : EUR {AI:.4f}")

# =============================================================================
# 6.  CVA engine  (identical to Q6)
# =============================================================================
def compute_risky_gross(cds_spread: float,
                        recovery:   float = 0.40) -> float:
    """
    Risky gross price = RF gross price - CVA.
    CVA = (1-R)*N * sum_i [Q(T_{i-1}) - Q(T_i)] * Z_rf(T_i)
    """
    lam    = cds_spread / (1.0 - recovery)
    lgd    = 1.0 - recovery
    cva    = 0.0
    Q_prev = 1.0

    for _, row in exp_df.iterrows():
        Q_i   = math.exp(-lam * row["tau_i"])
        dQ    = Q_prev - Q_i
        cva  += lgd * N * dQ * row["Z_rf"]
        Q_prev = Q_i

    return rf_npv_gross - cva

# =============================================================================
# 7.  Market price inputs
# =============================================================================
MKT_CLEAN_PCT  = 100.81          # % of par  (EuroTLX, 05-Nov-2025)
MKT_CLEAN_EUR  = N * MKT_CLEAN_PCT / 100.0    # EUR 1008.10
MKT_GROSS_EUR  = MKT_CLEAN_EUR + AI           # EUR 1012.879

CDS_MARKET_BP  = 72.0            # 5Y UniCredit CDS (investing.com)
RECOVERY_BASE  = 0.40

# Q6 risky value at 72 bp
risky_gross_q6 = compute_risky_gross(CDS_MARKET_BP / 1e4, RECOVERY_BASE)
risky_clean_q6 = risky_gross_q6 - AI

print(f"\nQ6 risky NPV   gross : EUR {risky_gross_q6:.4f}  "
      f"({risky_gross_q6 / 10:.4f}%)")
print(f"Q6 risky NPV   clean : EUR {risky_clean_q6:.4f}  "
      f"({risky_clean_q6 / 10:.4f}%)")
print(f"\nMarket clean price   : EUR {MKT_CLEAN_EUR:.4f}  "
      f"({MKT_CLEAN_PCT:.2f}%)")
print(f"Market gross price   : EUR {MKT_GROSS_EUR:.4f}")

# =============================================================================
# 8.  Price comparison table
# =============================================================================
print("\n" + "=" * 66)
print("Q10 -- Market Price Comparison")
print("=" * 66)
print(f"\n{'Price measure':<40} {'EUR':>9}  {'% par':>7}")
print("-" * 60)
print(f"{'RF fair value (DB model, Q5)':<40} "
      f"{rf_npv_clean:>9.2f}  {rf_npv_clean / 10:>7.2f}%")
print(f"{'Risky fair value (CVA-adj., Q6/Q8)':<40} "
      f"{risky_clean_q6:>9.2f}  {risky_clean_q6 / 10:>7.2f}%")
print(f"{'EuroTLX market price':<40} "
      f"{MKT_CLEAN_EUR:>9.2f}  {MKT_CLEAN_PCT:>7.2f}%")
print("-" * 60)
print(f"{'Market vs RF fair value':<40} "
      f"{MKT_CLEAN_EUR - rf_npv_clean:>+9.2f}  "
      f"{(MKT_CLEAN_PCT - rf_npv_clean / 10):>+7.2f}pp")
print(f"{'Market vs risky fair value':<40} "
      f"{MKT_CLEAN_EUR - risky_clean_q6:>+9.2f}  "
      f"{(MKT_CLEAN_PCT - risky_clean_q6 / 10):>+7.2f}pp")

# =============================================================================
# 9.  Market-implied CDS spread
# =============================================================================
# V_risky(s*) = MKT_GROSS_EUR  =>  solve for s*
# The function is monotonically decreasing in s.
# We first check whether a solution with s > 0 exists.

v_at_zero = compute_risky_gross(0.0, RECOVERY_BASE)
v_at_200  = compute_risky_gross(0.0200, RECOVERY_BASE)   # 200 bp

print(f"\nV_risky(s=0 bp)   = EUR {v_at_zero:.4f}  "
      f"(market gross = EUR {MKT_GROSS_EUR:.4f})")
print(f"V_risky(s=200 bp) = EUR {v_at_200:.4f}")

if v_at_zero < MKT_GROSS_EUR:
    # Market gross > RF gross => no positive root
    print("\nMarket price EXCEEDS risk-free model price.")
    print("No positive implied CDS spread exists.")
    print("The market prices the bond above the risk-free model value.")

    # Approximate implied yield spread via modified duration
    MOD_DUR    = 6.8          # approximate, consistent with LaTeX writeup
    delta_P    = (MKT_CLEAN_PCT - rf_npv_clean / 10) / 100.0
    delta_y_bp = -(delta_P / MOD_DUR) * 1e4

    print(f"\n  Delta_P (mkt - RF clean)    : {delta_P * 100:+.4f}%")
    print(f"  Modified duration (approx)  : {MOD_DUR:.1f} years")
    print(f"  Implied delta_y             : {delta_y_bp:+.1f} bp")
    print(f"  Market-implied CDS (approx) : {delta_y_bp:+.1f} bp "
          f"(effectively zero / negative)")

else:
    # Solve numerically for s*
    def objective(s):
        return compute_risky_gross(s, RECOVERY_BASE) - MKT_GROSS_EUR

    s_star = brentq(objective, 0.0, 0.50, xtol=1e-10)
    lam_star = s_star / (1.0 - RECOVERY_BASE)
    print(f"\nMarket-implied CDS spread : {s_star * 1e4:.2f} bp")
    print(f"Market-implied lambda     : {lam_star:.6f} p.a.")

# =============================================================================
# 10.  Implied spread sensitivity to recovery rate
# =============================================================================
print("\n" + "=" * 66)
print("Implied credit spread sensitivity to recovery rate")
print(f"  Market gross price = EUR {MKT_GROSS_EUR:.4f}")
print("=" * 66)
print(f"  {'Recovery':>10} {'Lambda':>12} {'CVA EUR':>10} "
      f"{'Risky Clean EUR':>16} {'Delta_y (bp)':>13}")
print("-" * 66)

MOD_DUR = 6.8
for rec in [0.20, 0.30, 0.40, 0.50, 0.60]:
    lam_r      = (CDS_MARKET_BP / 1e4) / (1.0 - rec)
    risky_g_r  = compute_risky_gross(CDS_MARKET_BP / 1e4, rec)
    risky_c_r  = risky_g_r - AI
    # Check for positive root
    v0         = compute_risky_gross(0.0, rec)
    if v0 < MKT_GROSS_EUR:
        delta_P   = (MKT_CLEAN_PCT - rf_npv_clean / 10) / 100.0
        delta_y   = -(delta_P / MOD_DUR) * 1e4
        s_impl_bp = delta_y
        s_label   = f"~{s_impl_bp:+.1f} bp (approx)"
    else:
        def obj_r(s, r=rec):
            return compute_risky_gross(s, r) - MKT_GROSS_EUR
        s_impl    = brentq(obj_r, 0.0, 0.50, xtol=1e-10)
        s_label   = f"{s_impl * 1e4:.2f} bp"

    marker = "  <-- base" if rec == RECOVERY_BASE else ""
    print(f"  {rec*100:>9.0f}%  {lam_r:>12.6f}  "
          f"{(rf_npv_gross - risky_g_r):>10.4f}  "
          f"{risky_c_r:>16.4f}  "
          f"{s_label:>13}{marker}")

# =============================================================================
# 11.  Bond-CDS basis
# =============================================================================
MOD_DUR   = 6.8
delta_P   = (MKT_CLEAN_PCT - rf_npv_clean / 10) / 100.0
delta_y   = -(delta_P / MOD_DUR) * 1e4          # bp; approx implied spread
basis_bp  = delta_y - CDS_MARKET_BP              # bond implied minus CDS

print("\n" + "=" * 66)
print("Bond-CDS Basis")
print("=" * 66)
print(f"  CDS market spread          : {CDS_MARKET_BP:+.2f} bp")
print(f"  Bond-implied spread (approx): {delta_y:+.1f} bp")
print(f"  Bond-CDS basis              : {basis_bp:+.1f} bp  "
      f"({'bond rich vs CDS' if basis_bp < 0 else 'bond cheap vs CDS'})")

# =============================================================================
# 12.  Summary table
# =============================================================================
print("\n" + "=" * 66)
print("FINAL SUMMARY  (EUR per 1000 nominal, trade date 05-Nov-2025)")
print("=" * 66)
print(f"{'Metric':<40} {'Value':>15}")
print("-" * 58)
print(f"{'RF clean price (Q5)':<40} "
      f"{rf_npv_clean / 10:>14.2f}%")
print(f"{'Risky clean price (Q6, CDS=72bp)':<40} "
      f"{risky_clean_q6 / 10:>14.2f}%")
print(f"{'EuroTLX market clean price':<40} "
      f"{MKT_CLEAN_PCT:>14.2f}%")
print(f"{'Market CDS spread':<40} "
      f"{'72 bp':>15}")
print(f"{'Bond-implied spread (approx)':<40} "
      f"{delta_y:>+14.1f} bp")
print(f"{'Bond-CDS basis':<40} "
      f"{basis_bp:>+14.1f} bp")
print(f"{'Market vs RF (price gap)':<40} "
      f"{MKT_CLEAN_EUR - rf_npv_clean:>+14.4f} EUR")
print(f"{'Market vs risky (price gap)':<40} "
      f"{MKT_CLEAN_EUR - risky_clean_q6:>+14.4f} EUR")

# =============================================================================
# 13.  Save outputs
# =============================================================================
summary = {
    "Metric": [
        "RF clean price (EUR)",
        "RF clean price (%)",
        "Risky clean price (EUR)",
        "Risky clean price (%)",
        "Market clean price (EUR)",
        "Market clean price (%)",
        "Accrued interest (EUR)",
        "CDS spread (bp)",
        "Bond-implied spread (bp)",
        "Bond-CDS basis (bp)",
        "Market vs RF gap (EUR)",
        "Market vs risky gap (EUR)",
    ],
    "Value": [
        round(rf_npv_clean, 4),
        round(rf_npv_clean / 10, 4),
        round(risky_clean_q6, 4),
        round(risky_clean_q6 / 10, 4),
        round(MKT_CLEAN_EUR, 4),
        round(MKT_CLEAN_PCT, 4),
        round(AI, 4),
        round(CDS_MARKET_BP, 2),
        round(delta_y, 2),
        round(basis_bp, 2),
        round(MKT_CLEAN_EUR - rf_npv_clean, 4),
        round(MKT_CLEAN_EUR - risky_clean_q6, 4),
    ],
}

pd.DataFrame(summary).to_csv("q10_summary.csv", index=False)
print("\nSaved: q10_summary.csv")

Trade date : November 5th, 2025   Spot date : November 7th, 2025

Risk-free NPV  gross : EUR 1119.2599  (111.9260%)
Risk-free NPV  clean : EUR 1114.4805  (111.4480%)
Accrued interest     : EUR 4.7794

Q6 risky NPV   gross : EUR 1066.0983  (106.6098%)
Q6 risky NPV   clean : EUR 1061.3188  (106.1319%)

Market clean price   : EUR 1008.1000  (100.81%)
Market gross price   : EUR 1012.8794

Q10 -- Market Price Comparison

Price measure                                  EUR    % par
------------------------------------------------------------
RF fair value (DB model, Q5)               1114.48   111.45%
Risky fair value (CVA-adj., Q6/Q8)         1061.32   106.13%
EuroTLX market price                       1008.10   100.81%
------------------------------------------------------------
Market vs RF fair value                    -106.38   -10.64pp
Market vs risky fair value                  -53.22    -5.32pp

V_risky(s=0 bp)   = EUR 1119.2599  (market gross = EUR 1012.8794)
V_risky(s=200 bp) = EUR 